In [8]:
#requirements
# pip install cdrc
# pip install httpx
# pip install pydantic_settings
#pip install ImageMagic
#CDR Schemas import
#!pip install git+https://github.com/DARPA-CRITICALMAAS/cdr_schemas.git


In [10]:
# CDR AUtentification
import cdrc
client = cdrc.CDRClient(token="") #Add API -------------------------------------------------------

In [11]:
from pathlib import Path
from pydantic_settings import BaseSettings
import httpx
from cdr_schemas.cdr_responses.prospectivity import ProspectModelMetaData
from cdr_schemas.prospectivity_input import ProspectivityOutputLayer
from json import load as json_load

In [12]:
class Settings(BaseSettings):
    # TO BE CHANGED BY TA3-4 system.
    system_name: str = "Beak_SOM"                                              #------------------------
    system_version: str = "0.0.1_BASELINE_BISON"                               #------------------------
    ml_model_name: str = "BASELINE_BISON"                                      #------------------------
    ml_model_version: str = "F14_X50_Y50_E50_CMAX50_20240722-074734"           #------------------------


    # To be provided to TA3-4 system by CDR admin
    user_api_token: str = "" #Add API --------------------------------------------------------------
    cdr_host: str = "https://api.cdr.land"
    admin_cdr_host: str = "https://admin.cdr.land"
    # cdr_host: str = "http://0.0.0.0:8333"
    # admin_cdr_host: str = "http://0.0.0.0:3333"

    class Config:
        case_sensitive = False
        env_file = ".env"
        env_file_encoding = "utf-8"

def send_outputs(output_type: str, output_path: Path, payload, app_settings):
    print(f"Sending {output_path.name} to CDR...")

    # checks outputs type set properly
    assert output_type in ["likelihood", "uncertainty"]
    # checks outputs file exists
    assert output_path.is_file()

    #  create output layer metadata
    results = ProspectivityOutputLayer(**{
        "system": app_settings.system_name,
        "system_version": app_settings.system_version,
        "model": app_settings.ml_model_name,
        "model_version": app_settings.ml_model_version,
        "model_run_id": payload.model_run_id,
        "cma_id": payload.cma.cma_id,
        "output_type": output_type,
        "title": output_path.name
    })
    files_ = {"input_file": (
        output_path.name, 
        open(output_path, "rb"), "application/octet-stream")
    }

    # prepare the CDR client
    headers = {'Authorization': f'Bearer {app_settings.user_api_token}'}
    #client = httpx.Client(follow_redirects=True)
    client = httpx.Client(follow_redirects=True, trust_env=False)

    # post the request
    resp = client.post(
        url=f"{app_settings.cdr_host}/v1/prospectivity/prospectivity_output_layer",
        data={"metadata": results.model_dump_json(exclude_none=True)},
        files=files_,
        headers=headers,
    
    )
    if resp.status_code != 200 or resp.status_code != 204:
        print(f"An Error Occurred sending {output_path.name} to CDR.")
        print(resp.text)
    else:
        print(f"Finished sending {output_path.name} to CDR.")

In [13]:
# instantiates variables
send_settings = Settings()

#send_file_path = Path("D:\\JN\\CMAAS\\BMU_BMU_label_count.tif") #-------------------------------------------
send_file_path = Path("BMU_BMU_label_count.tif") #-------------------------------------------

#with open("D:\\JN\\CMAAS\\BEAK_MVT.json", "r") as f:           #-------------------------------------
with open("BEAK_MVT.json", "r") as f:           #-------------------------------------
    prospectivity_model_run_event = json_load(f)
    payload_dict = prospectivity_model_run_event.get("payload")

payload_obj = ProspectModelMetaData(
    model_run_id = payload_dict.get("model_run_id"),
    cma = payload_dict.get("cma"),
    model_type = payload_dict.get("model_type"),
    train_config = payload_dict.get("train_config"),
    evidence_layers = payload_dict.get("evidence_layers"),
)

In [ ]:
#Note: Ignore the error on the next cell, the layer is being uploaded check the layer at https://api.cdr.land/docs#/

In [14]:
# makes reqeust
send_outputs(
    output_type="likelihood",   #------------------------------------
    output_path=send_file_path,
    payload=payload_obj,
    app_settings=send_settings
)

Sending BMU_BMU_label_count.tif to CDR...


ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:997)